# 01 · Cắt keyframe và dựng index — **Người 1 (nhóm trưởng)**

Chạy **đầu tiên**. Ba notebook lab (02, 03, 04) đều đọc file zip do notebook này tạo ra.

> **Chạy được ngay, không cần chuẩn bị gì.** Bảng fps của BTC đã nhúng sẵn trong ô
> SECTION 02 — không phải đẩy file nào lên Drive.

| | |
|---|---|
| **Đầu vào** | `MyDrive/aic25/AIC_Video/*.mp4` |
| **Đầu ra** | `MyDrive/aic26/output_demo/v004/` → `zips/`, `siglip.index`, `clip.index`, `keyframe_metadata.json` |
| **Model** | TransNetV2 · SigLIP-large-384 · CLIP ViT-H/14 |

## Cách chạy

**Lần đầu — thử 3 video, khoảng 30 phút.** `SO_VIDEO_THU = 3` đã đặt sẵn ở SECTION 02.

```
SECTION 00 → 01 → 02 → 02b → 03 → 04 → 06 → 07 → 08 → 09 → 10 → 11 → 12
```

Bỏ qua **SECTION 05** (đã tắt) và **SECTION 07b** (ô thử một video, không cần cho lần chạy thật).

Xong thì kiểm năm thứ:

| Kiểm | Đúng thì thấy |
|---|---|
| Zip | `zips/AIC2026_frames_001.zip` trên Drive |
| Index | `siglip.index` và `clip.index` trong `indexes/` |
| Số vector | SECTION 09 in `total N` — bằng tổng keyframe của 3 video |
| Metadata | `keyframe_metadata.json` có `frame_idx`, `fps`, `width`, `height` hợp lý |
| Tiến độ | `progress.json`: `done_videos` có 3 tên |

**Chạy thật.** Đặt `SO_VIDEO_THU = 0` ở SECTION 02, chạy lại SECTION 03, rồi 07 → 09 → 10 → 11 → 12.

Ba video đã cắt không bị làm lại — `todo_list` tự loại tên có trong `done_videos`.
Colab ngắt giữa chừng cũng không sao: mở lại, chạy lại, nó tiếp tục từ chỗ dở.

Ước tính 50 video L-series: **4–7 giờ cắt + 1,5–2 giờ index**, khoảng 10 GB.

## Cách cắt đã chốt

```
TransNetV2 → cảnh → lấy mẫu THEO TỐC ĐỘ → lưu JPEG ở độ phân giải GỐC
```

| | |
|---|---|
| **Q1** lấy mẫu | `n = 2` tới 1,67 giây, sau đó cứ 2 giây thêm 1 ảnh, kẹp [2, 40], lùi biên 3 frame.<br>@30fps: `8–50 → 2 ảnh · 51–110 → 3 · 111–170 → 4` |
| **Q2** lọc trùng | **bỏ hẳn ở bước cắt** — paper §3.2.3 lọc sau khi trích đặc trưng, dùng chính embedding nạp FAISS. Bỏ luôn được CLIP ViT-B/32 (SECTION 05) |
| **Q3** độ phân giải | **giữ nguyên gốc**. Thu nhỏ xuống 960 làm hỏng OCR rõ rệt, phóng to vô ích |
| **fps** | lấy từ bảng BTC, **không dò từ video** — ffprobe có thể trả 30.0 ở video BTC ghi 29.97 |

Chi tiết và số đo: [KIEN_TRUC_PIPELINE.md](../../KIEN_TRUC_PIPELINE.md) mục 8 và 9.5.

> **Nguồn:** tách từ `(thanhbangcao)FULL_PIPELINE_AIC_2026.ipynb`.
> Mỗi notebook chạy trong **một phiên Colab riêng** — thư viện giữa các notebook xung đột nhau.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install Dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install] Installing baseline dependencies...")
_pip("--upgrade", "transnetv2-pytorch==1.0.5", "ffmpeg-python")
_pip("open_clip_torch")
_pip("faiss-cpu")   # faiss-gpu trên PyPI dừng ở 2022, không cài được
_pip("tqdm")
_pip("transformers>=4.40.0", "Pillow", "opencv-python-headless")
_pip("gdown")
print("[Install] All done\n")


[Install] Installing baseline dependencies...
[Install] All done



In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive  ← PHẢI mount TRƯỚC mọi thứ liên quan /content/drive
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os
DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted OK.")


Mounted at /content/drive
[Drive] Mounted OK.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Config & Dirs  ← SAU khi Drive đã mount
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import sys, json, glob, time, math, zipfile, shutil, subprocess as _sp
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─── MODE ────────────────────────────────────────────────────────────────────
# "TEST"  → chỉ xử lý TEST_VIDEOS (~83 videos, đủ để kiểm thử pipeline)
# "FULL"  → toàn bộ ~1478 videos (production run)
MODE = "TEST"   # ← đổi thành "FULL" khi cần chạy thật

# Chạy thử toàn chuỗi trên vài video trước khi chạy hết. 0 = chạy hết.
# Lần đầu để 3: cắt → zip → index xong trong ~30 phút, đủ để kiểm mọi khâu.
SO_VIDEO_THU = 3

# ─── PATHS (path cứng, không đổi) ────────────────────────────────────────────
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = "/content/drive/MyDrive/aic26/output_demo/v004"
# Weights là tài sản DÙNG CHUNG, cố ý đặt NGOÀI output_demo/v004 để đổi
# version không phải tải lại vài trăm MB.
SHARED_MODELS_DIR = "/content/drive/MyDrive/aic26"
Path(SHARED_MODELS_DIR).mkdir(parents=True, exist_ok=True)
CKPT_TNV2    = f"{SHARED_MODELS_DIR}/transnetv2-pytorch-weights.pth"
LOCAL_KF_DIR = "/content/kf_buffer"

# ─── FPS: BẢNG CỦA BTC, NHÚNG THẲNG VÀO ĐÂY ─────────────────────────────────
# Trích từ data_raw/map-keyframes-aic25-b1 — 873 video L21–L30, gom theo giá trị.
# Nhúng vào notebook để KHÔNG phải đẩy file nào lên Drive trước khi chạy.
#
# Vì sao không dò bằng ffprobe: BTC dùng CHÍNH bảng này để đánh số frame trong
# đáp án. ffprobe có thể trả 30.0 ở video mà BTC ghi 29.97 — lệch 20%, tới phút
# thứ 20 là sai gần 4 phút, đủ để trượt hẳn cảnh chứa đáp án.
_FPS_NHOM = {
    25.0: (
        "L21_V003 L21_V008 L21_V009 L21_V010 L21_V011 L21_V017 L21_V018 L21_V024 L21_V025 "
        "L21_V031 L22_V002 L22_V003 L22_V007 L22_V008 L22_V009 L22_V010 L22_V016 L22_V017 "
        "L22_V023 L22_V024 L22_V030 L22_V031 L23_V001 L23_V002 L23_V003 L23_V004 L23_V005 "
        "L23_V006 L23_V007 L23_V008 L23_V009 L23_V010 L23_V011 L23_V012 L23_V013 L23_V014 "
        "L23_V015 L23_V016 L23_V017 L23_V018 L23_V019 L23_V020 L23_V021 L23_V022 L23_V023 "
        "L23_V024 L23_V025 L24_V002 L24_V003 L24_V004 L24_V005 L24_V006 L24_V007 L24_V008 "
        "L24_V009 L24_V010 L24_V011 L24_V012 L24_V013 L24_V014 L24_V015 L24_V016 L24_V018 "
        "L24_V026 L24_V032 L24_V043 L25_V001 L25_V002 L25_V003 L25_V006 L25_V007 L25_V009 "
        "L25_V010 L25_V011 L25_V012 L25_V015 L25_V016 L25_V018 L25_V019 L25_V020 L25_V023 "
        "L25_V024 L25_V026 L25_V027 L25_V028 L25_V029 L25_V032 L25_V033 L25_V035 L25_V036 "
        "L25_V037 L25_V038 L25_V041 L25_V042 L25_V044 L25_V045 L25_V046 L25_V047 L25_V050 "
        "L25_V051 L25_V053 L25_V054 L25_V055 L25_V056 L25_V059 L25_V060 L25_V062 L25_V063 "
        "L25_V064 L25_V065 L25_V068 L25_V069 L25_V071 L25_V072 L25_V073 L25_V075 L25_V076 "
        "L25_V079 L25_V080 L25_V081 L25_V082 L25_V083 L25_V086 L25_V087 L26_V001 L26_V002 "
        "L26_V003 L26_V004 L26_V005 L26_V006 L26_V007 L26_V008 L26_V009 L26_V010 L26_V011 "
        "L26_V012 L26_V013 L26_V014 L26_V015 L26_V016 L26_V017 L26_V018 L26_V019 L26_V020 "
        "L26_V021 L26_V022 L26_V023 L26_V024 L26_V025 L26_V026 L26_V027 L26_V028 L26_V029 "
        "L26_V030 L26_V031 L26_V032 L26_V033 L26_V034 L26_V035 L26_V036 L26_V037 L26_V038 "
        "L26_V039 L26_V040 L26_V041 L26_V042 L26_V043 L26_V044 L26_V045 L26_V046 L26_V047 "
        "L26_V048 L26_V049 L26_V050 L26_V051 L26_V052 L26_V053 L26_V054 L26_V055 L26_V056 "
        "L26_V057 L26_V058 L26_V059 L26_V060 L26_V061 L26_V062 L26_V063 L26_V064 L26_V065 "
        "L26_V066 L26_V067 L26_V068 L26_V069 L26_V070 L26_V071 L26_V072 L26_V073 L26_V074 "
        "L26_V075 L26_V076 L26_V077 L26_V078 L26_V079 L26_V080 L26_V081 L26_V082 L26_V083 "
        "L26_V084 L26_V085 L26_V086 L26_V087 L26_V088 L26_V089 L26_V090 L26_V091 L26_V092 "
        "L26_V093 L26_V094 L26_V095 L26_V096 L26_V097 L26_V098 L26_V099 L26_V100 L26_V101 "
        "L26_V102 L26_V103 L26_V104 L26_V105 L26_V106 L26_V107 L26_V108 L26_V109 L26_V110 "
        "L26_V111 L26_V112 L26_V113 L26_V114 L26_V115 L26_V116 L26_V117 L26_V118 L26_V119 "
        "L26_V120 L26_V121 L26_V122 L26_V123 L26_V124 L26_V125 L26_V126 L26_V127 L26_V128 "
        "L26_V129 L26_V130 L26_V131 L26_V132 L26_V133 L26_V134 L26_V135 L26_V136 L26_V137 "
        "L26_V138 L26_V139 L26_V140 L26_V141 L26_V142 L26_V143 L26_V144 L26_V145 L26_V146 "
        "L26_V147 L26_V148 L26_V149 L26_V150 L26_V151 L26_V152 L26_V153 L26_V154 L26_V155 "
        "L26_V156 L26_V157 L26_V158 L26_V159 L26_V160 L26_V161 L26_V162 L26_V163 L26_V164 "
        "L26_V165 L26_V166 L26_V167 L26_V168 L26_V169 L26_V170 L26_V171 L26_V172 L26_V173 "
        "L26_V174 L26_V175 L26_V176 L26_V177 L26_V178 L26_V179 L26_V180 L26_V181 L26_V182 "
        "L26_V183 L26_V184 L26_V185 L26_V186 L26_V187 L26_V188 L26_V189 L26_V190 L26_V191 "
        "L26_V192 L26_V193 L26_V194 L26_V195 L26_V196 L26_V197 L26_V198 L26_V199 L26_V200 "
        "L26_V201 L26_V202 L26_V203 L26_V204 L26_V205 L26_V206 L26_V207 L26_V208 L26_V209 "
        "L26_V210 L26_V211 L26_V212 L26_V213 L26_V214 L26_V215 L26_V216 L26_V217 L26_V218 "
        "L26_V219 L26_V220 L26_V221 L26_V222 L26_V223 L26_V224 L26_V225 L26_V226 L26_V227 "
        "L26_V228 L26_V229 L26_V230 L26_V231 L26_V232 L26_V233 L26_V234 L26_V235 L26_V236 "
        "L26_V237 L26_V238 L26_V239 L26_V240 L26_V241 L26_V242 L26_V243 L26_V244 L26_V245 "
        "L26_V246 L26_V247 L26_V248 L26_V249 L26_V250 L26_V251 L26_V252 L26_V253 L26_V254 "
        "L26_V255 L26_V256 L26_V257 L26_V258 L26_V259 L26_V260 L26_V261 L26_V262 L26_V263 "
        "L26_V264 L26_V265 L26_V266 L26_V267 L26_V268 L26_V269 L26_V270 L26_V271 L26_V272 "
        "L26_V273 L26_V274 L26_V275 L26_V276 L26_V277 L26_V278 L26_V279 L26_V280 L26_V281 "
        "L26_V282 L26_V283 L26_V284 L26_V285 L26_V286 L26_V287 L26_V288 L26_V289 L26_V290 "
        "L26_V291 L26_V292 L26_V293 L26_V294 L26_V295 L26_V296 L26_V297 L26_V298 L26_V299 "
        "L26_V300 L26_V301 L26_V302 L26_V303 L26_V304 L26_V305 L26_V306 L26_V307 L26_V308 "
        "L26_V309 L26_V310 L26_V311 L26_V312 L26_V313 L26_V314 L26_V315 L26_V316 L26_V317 "
        "L26_V318 L26_V319 L26_V320 L26_V321 L26_V322 L26_V323 L26_V324 L26_V325 L26_V326 "
        "L26_V327 L26_V328 L26_V329 L26_V330 L26_V331 L26_V332 L26_V333 L26_V334 L26_V335 "
        "L26_V336 L26_V337 L26_V338 L26_V339 L26_V340 L26_V341 L26_V342 L26_V343 L26_V344 "
        "L26_V345 L26_V346 L26_V347 L26_V348 L26_V349 L26_V350 L26_V351 L26_V352 L26_V353 "
        "L26_V354 L26_V355 L26_V356 L26_V357 L26_V358 L26_V359 L26_V360 L26_V361 L26_V362 "
        "L26_V363 L26_V364 L26_V365 L26_V366 L26_V367 L26_V368 L26_V369 L26_V370 L26_V371 "
        "L26_V372 L26_V373 L26_V374 L26_V375 L26_V376 L26_V377 L26_V378 L26_V379 L26_V380 "
        "L26_V381 L26_V382 L26_V383 L26_V384 L26_V385 L26_V386 L26_V387 L26_V388 L26_V389 "
        "L26_V390 L26_V391 L26_V392 L26_V393 L26_V394 L26_V395 L26_V396 L26_V397 L26_V398 "
        "L26_V399 L26_V400 L26_V401 L26_V402 L26_V403 L26_V404 L26_V405 L26_V406 L26_V407 "
        "L26_V408 L26_V409 L26_V410 L26_V411 L26_V412 L26_V413 L26_V414 L26_V415 L26_V416 "
        "L26_V418 L26_V419 L26_V420 L26_V421 L26_V422 L26_V423 L26_V424 L26_V425 L26_V426 "
        "L26_V427 L26_V428 L26_V429 L26_V430 L26_V431 L26_V432 L26_V433 L26_V434 L26_V435 "
        "L26_V436 L26_V437 L26_V438 L26_V439 L26_V440 L26_V441 L26_V442 L26_V443 L26_V444 "
        "L26_V445 L26_V446 L26_V447 L26_V448 L26_V449 L26_V450 L26_V451 L26_V452 L26_V453 "
        "L26_V454 L26_V455 L26_V456 L26_V457 L26_V458 L26_V459 L26_V460 L26_V461 L26_V462 "
        "L26_V463 L26_V464 L26_V465 L26_V466 L26_V467 L26_V468 L26_V469 L26_V470 L26_V471 "
        "L26_V472 L26_V473 L26_V474 L26_V475 L26_V476 L26_V477 L26_V478 L26_V479 L26_V480 "
        "L26_V481 L26_V482 L26_V483 L26_V484 L26_V485 L26_V486 L26_V487 L26_V488 L26_V489 "
        "L26_V490 L26_V491 L26_V492 L26_V493 L26_V494 L26_V495 L26_V496 L26_V497 L26_V498 "
        "L26_V499 L27_V001 L27_V002 L27_V003 L27_V004 L27_V005 L27_V006 L27_V007 L27_V008 "
        "L27_V009 L27_V010 L27_V011 L27_V012 L27_V013 L27_V014 L27_V015 L27_V016 L28_V001 "
        "L28_V002 L28_V003 L28_V004 L28_V005 L28_V006 L28_V007 L28_V008 L28_V009 L28_V010 "
        "L28_V011 L28_V012 L28_V013 L28_V014 L28_V015 L28_V016 L28_V017 L28_V018 L28_V019 "
        "L28_V020 L28_V021 L28_V022 L28_V023 L28_V024 L29_V001 L29_V002 L29_V003 L29_V004 "
        "L29_V005 L29_V006 L29_V007 L29_V008 L29_V009 L29_V010 L29_V011 L29_V012 L29_V013 "
        "L29_V014 L29_V015 L29_V016 L29_V017 L29_V018 L29_V019 L29_V020 L29_V021 L29_V022 "
        "L29_V023 L30_V001 L30_V002 L30_V003 L30_V004 L30_V005 L30_V006 L30_V007 L30_V008 "
        "L30_V009 L30_V010 L30_V011 L30_V012 L30_V013 L30_V014 L30_V015 L30_V016 L30_V017 "
        "L30_V018 L30_V019 L30_V020 L30_V021 L30_V022 L30_V023 L30_V024 L30_V025 L30_V026 "
        "L30_V027 L30_V028 L30_V029 L30_V030 L30_V031 L30_V032 L30_V033 L30_V034 L30_V035 "
        "L30_V036 L30_V037 L30_V038 L30_V039 L30_V040 L30_V041 L30_V042 L30_V043 L30_V044 "
        "L30_V045 L30_V046 L30_V047 L30_V048 L30_V049 L30_V050 L30_V051 L30_V052 L30_V053 "
        "L30_V054 L30_V055 L30_V056 L30_V057 L30_V058 L30_V059 L30_V060 L30_V061 L30_V062 "
        "L30_V063 L30_V064 L30_V065 L30_V066 L30_V067 L30_V068 L30_V069 L30_V070 L30_V071 "
        "L30_V072 L30_V073 L30_V074 L30_V075 L30_V076 L30_V077 L30_V078 L30_V079 L30_V080 "
        "L30_V081 L30_V082 L30_V083 L30_V084 L30_V085 L30_V086 L30_V087 L30_V088 L30_V089 "
        "L30_V090 L30_V091 L30_V092 L30_V093 L30_V094 L30_V095 L30_V096 "
    ),
    26.44: (
        "L24_V044 "
    ),
    29.97: (
        "L25_V004 L25_V005 L25_V008 L25_V013 L25_V014 L25_V017 L25_V021 L25_V022 L25_V025 "
        "L25_V030 L25_V031 L25_V034 L25_V039 L25_V040 L25_V043 L25_V048 L25_V049 L25_V052 "
        "L25_V057 L25_V058 L25_V061 L25_V066 L25_V067 L25_V070 L25_V074 L25_V077 L25_V078 "
        "L25_V084 L25_V085 L25_V088 "
    ),
    30.0: (
        "L21_V001 L21_V002 L21_V005 L21_V006 L21_V007 L21_V012 L21_V013 L21_V014 L21_V015 "
        "L21_V016 L21_V019 L21_V021 L21_V022 L21_V023 L21_V026 L21_V027 L21_V028 L21_V029 "
        "L21_V030 L22_V001 L22_V004 L22_V005 L22_V006 L22_V011 L22_V012 L22_V013 L22_V014 "
        "L22_V015 L22_V018 L22_V019 L22_V020 L22_V021 L22_V022 L22_V025 L22_V026 L22_V027 "
        "L22_V028 L22_V029 L24_V017 L24_V019 L24_V020 L24_V021 L24_V022 L24_V023 L24_V024 "
        "L24_V025 L24_V027 L24_V028 L24_V029 L24_V030 L24_V031 L24_V033 L24_V035 L24_V036 "
        "L24_V037 L24_V038 L24_V039 L24_V040 L24_V041 L24_V042 L24_V045 "
    ),
}
FPS_MAP: dict[str, float] = {v: f for f, s in _FPS_NHOM.items() for v in s.split()}

# Batch 2 về sau: đẩy fps_map.json mới lên đường dẫn dưới, nó sẽ ghi đè bảng trên.
_FPS_THEM = f"{SHARED_MODELS_DIR}/fps_map.json"
if os.path.exists(_FPS_THEM):
    with open(_FPS_THEM) as f:
        FPS_MAP.update({k: float(v) for k, v in json.load(f).items()})
    print(f"[fps] Có file bổ sung trên Drive → tổng {len(FPS_MAP)} video")

from collections import Counter as _C
print(f"[fps] {len(FPS_MAP)} video · " +
      " · ".join(f"{k}fps: {v}" for k, v in sorted(_C(FPS_MAP.values()).items())))

# ─── DANH SÁCH VIDEO ─────────────────────────────────────────────────────────
# Nguồn: Danh_gia_Query_AIC2025.md → ground_truth_2025.json (cùng thư mục).
#
# Chia theo ĐỘ TIN CẬY CỦA ĐÁP ÁN, không theo vòng thi.
# (Bản trước chia theo "_GT_L = Round 1" / "_GT_K = Round 2" — cách chia đó sai:
#  L-series có cả câu R2/R3, và toàn bộ Round 3 bị bỏ sót vì lúc đó chưa làm xong
#  đáp án batch 3.)
#
#   _GT_FRAME       63 video – có video + frame, đáp án không bị ghi chú nghi ngờ
#                      → đo được CẢ video-level lẫn frame-level
#   _GT_VIDEO_ONLY   4 video – tài liệu chỉ ghi "Vid đúng là X", không có frame
#                      → CHỈ đo được video-level
#   _NEAR_MISS       2 video – tác giả tìm nhầm, tự ghi "Chưa tìm được video đúng"
#                      → KHÔNG phải đáp án, nhưng giống đề → hard negative rất tốt
#   _DIVERSITY      50 video – nhiễu chung, không liên quan câu nào
#
# KHÔNG có video ở đây: 16 câu chưa có đáp án
#   (R2-16, R2-19, R3-3, R3-5, R3-6, R3-9, R3-14, R3-19, R3-21, R3-22, R3-25, R3-27, R3-29, R3-30, R3-31, R3-34)
#   → đây là "bộ test không đáp án"; không biết video nên không cắt trước được,
#     phải chạy MODE="FULL" mới đánh giá được. Xem _NO_ANSWER_QUERIES bên dưới.

# ── Có đáp án đầy đủ (video + frame) ─────────────────────────────────────────
_GT_FRAME = {
    "K01_V009",  # R2-4 KIS: cứu hộ lũ, cụ bà đội nón lá trên xuồng cam
    "K01_V012",  # R3-24 TRAKE: giới thiệu du lịch, trong đoạn video có một người trả lời phỏn…
    "K01_V018",  # R2-15 KIS: cô gái phỏng vấn + Deadpool + múa đao
    "K02_V005",  # R2-5 KIS: triển lãm nghệ thuật robot vẽ tranh + sách MILK
    "K02_V007",  # R3-33 KIS: gồm các hình ảnh từ tài liệu văn bản/inforgraphic của tổng cục…
    "K02_V012",  # R3-35 QA: việc chế tạo vaccine phòng một loại virus. Thông tin trong đoạ…
    "K03_V019",  # R2-1 KIS: nhiều người áo cờ đỏ sao vàng trước biểu tượng con cua
    "K03_V023",  # R2-8 KIS: cầu vòm thép đỏ → ca nô vệt sóng tròn
    "K04_V013",  # R2-23 KIS: gấu bông xanh trong hộp trụ / cảm biến chuyển động
    "K04_V021",  # R3-16 KIS: Nhiều con rùa có kích thước nhỏ đang ăn rau cải trong một tran…
    "K05_V013",  # R2-24 KIS: ngư dân Campuchia nâng cá lớn nguy cấp
    "K05_V018",  # R2-25 KIS: robot 4 chân bánh xe trên đồng cỏ + bò
    "K05_V025",  # R2-7 KIS: người tuyết khổng lồ đội mũ + bình gốm băng tuyết
    "K06_V010",  # R2-20 KIS: cào muối đồng, đoàn người, "Gừng cay muối mặn"
    "K06_V014",  # R3-23 KIS: quay một phụ nữ cosplay nữ siêu nhân đang đi trong hành lang b…
    "K06_V022",  # R3-20 QA: Một cái bàn có để nhiều hộp đĩa CD. Một trong những bìa đĩa có…
    "K07_V025",  # R2-10 TRAKE: xưởng lắp ráp robot + công nhân + cửa ô tô
    "K07_V030",  # R2-17 KIS: phụ nữ đội nón len đeo kính "Happy New Year"
    "K08_V019",  # R3-7 KIS: quay từ phía sau một cậu bé và một cô gái đứng cạnh nhau. cậu …
    "K08_V021",  # R3-12 KIS: múa rồng, những người biểu diễn múa rồng mặc trang phục của cá…
    "K09_V007",  # R2-30 KIS: kị sĩ cưỡi ngựa đồng cỏ xanh / cờ đỏ
    "K09_V018",  # R2-6 KIS: đài phun nước trăm ngọn nến / phụ nữ bên ly nến
    "K10_V002",  # R3-4 KIS: quay các con động vật nối đuôi nhau. Con đi đầu lớn nhất, các …
    "K12_V001",  # R2-9 KIS: cá nhám 211kg treo trên xe cẩu
    "K12_V007",  # R3-28 KIS: tin tức về một cánh tay robot. Trong đoạn video có một mô phỏn…
    "K13_V003",  # R3-26 KIS: một tuyến đường ngập nước, người dân đi lại khó khăn. Đoạn vid…
    "K14_V027",  # R2-2 KIS: lễ hội đèn lồng, hanbok, đèn lồng khổng lồ hình người
    "K16_V004",  # R3-8 KIS: 3D mô phỏng nhiệm vụ của một tàu vũ trụ. Có đoạn mô phỏng tàu …
    "K16_V010",  # R2-14 KIS: icon hình não 5×10 / thống kê tác dụng phụ thuốc
    "K17_V003",  # R2-3 QA: bản đồ Việt Nam trái cây / Khu Du Lịch Quốc Gia Mộc Châu
    "K18_V002",  # R2-28 KIS: nhiều loài cá cảnh, cá đĩa, cá rồng, cá la hán
    "K19_V022",  # R3-11 KIS: Hai cảnh sát giao thông đang dùng bình chữa cháy để dập một ch…
    "K20_V009",  # R3-10 KIS: Một người phụ nữ đang bế một chú chó trả lời phỏng vấn. Chú ch…
    "L21_V015",  # R1-1 KIS: 4 phi hành gia mặc áo đen / tàu vũ trụ tư nhân
    "L21_V027",  # R1-14 KIS: điêu khắc cát, khói hồng
    "L21_V029",  # R1-2 KIS: đàn hổ con quý hiếm miền Nam
    "L22_V022",  # R1-23 KIS: thị trấn ven biển / cá mập (phim Spielberg 1975)
    "L22_V030",  # R1-8 KIS: lễ hội ẩm thực Nhật, cô bé đeo bạch tuộc đỏ
    "L23_V007",  # R1-24 KIS: đua xe đạp, 3 tay đua đồng phục trắng-vàng-xanh
    "L23_V017",  # R1-25 KIS: flycam đua xe đạp, áo xanh dương trắng vượt lên
    "L24_V033",  # R1-16 TRAKE: múa lân vàng đen trắng / lân quay cột
    "L25_V058",  # R2-21 QA: bài tập vận dụng, khoảng cách AB 2 chữ số, đáp C
    "L26_V004",  # R1-20 KIS: panna cotta 3 ly / nho đỏ bạc hà
    "L26_V056",  # R1-6 KIS: gỏi cuốn chay, bánh tráng vàng tím
    "L26_V072",  # R1-18 TRAKE: cắt nấm → củ năng → đậu hũ → bếp
    "L26_V176",  # R3-1 TRAKE: nấu cá, khuấy sốt → rưới nồi
    "L26_V178",  # R1-22 QA: bánh ít trần, 200g thịt nạc xay
    "L26_V194",  # R1-4 TRAKE: bột → măng tây → dầu → chiên
    "L26_V200",  # R1-12 KIS: trang trí bánh rán, chocolate + chuối + dâu
    "L26_V222",  # R3-15 KIS: một đoạn video nấu ăn một món ăn về tôm. Cảnh quay đang lột vỏ…
    "L26_V444",  # R3-32 QA: quay một món ăn được bày trên 1 vỉ nướng đặt trên bếp ga. Tuy …
    "L27_V010",  # R1-19 QA: đình thần Nguyễn Trung Trực Kiên Giang / 2 câu thơ
    "L27_V013",  # R1-9 KIS: thu hoạch dứa miền Tây, bà cụ + cô gái áo hồng + ghe xanh
    "L27_V014",  # R1-5 KIS: cho dê ăn, áo thun trắng + áo kẻ sọc tím
    "L29_V020",  # R3-2 KIS: miếng thiếc nhiều lỗ thủng do đinh đóng
    "L29_V023",  # R1-7 KIS: chim đầu đen ánh xanh mắt đỏ, rừng Nam Bộ
    "L30_V014",  # R2-11 KIS: phụ nữ lớn tuổi đọc sách / bìa ảnh cụ lão
    "L30_V017",  # R1-10 KIS: 3 người chơi nhạc cụ kim loại tròn / kệ sách
    "L30_V040",  # R2-18 KIS: mèo trong phòng, tô kim loại úp, bảng tên "Nà Ní"
    "L30_V057",  # R1-11 KIS: mảnh bìa đổ bóng thành chân dung / mũ lưỡi trai đen
    "L30_V072",  # R1-15 QA: câu lạc bộ FANA trao quà Xã Giang Ly Khánh Hòa
    "L30_V092",  # R1-17 KIS: trao quà từ thiện bệnh viện, phông đỏ COVID-19
    "L30_V095",  # R1-13 KIS: vệ sinh máy ảnh, ống kính + khăn tím hồng + tăm bông
}

# ── Chỉ biết video đúng, chưa có frame ───────────────────────────────────────
_GT_VIDEO_ONLY = {
    "K03_V022",  # R2-29 KIS: cảng biển, cầu dài ra biển, người đội nón tai bèo
    "K09_V012",  # R2-26 KIS: phỏng vấn Trump + đoàn người ven biển vác bao trắng
    "K09_V015",  # R2-13 KIS: bánh giá 9€ và 4.80€ / sản xuất 12 cái/tầng
    "K15_V010",  # R2-27 KIS: họp giá điện, màn hình LED chữ trắng nền xanh
}

# ── Tác giả tìm nhầm — giữ lại làm hard negative ─────────────────────────────
_NEAR_MISS = {
    "K07_V019",  # R2-12 KIS: lễ hội trình diễn cổ phục Nhật Bình / người mẫu
    "L26_V385",  # R1-6 KIS: gỏi cuốn chay (video tìm được, dù L26_V056 đúng hơn)
}

# ── Nhiễu: video không liên quan câu hỏi nào ─────────────────────────────────
# Chọn video đầu mỗi batch (ít khả năng chứa nội dung GT).
# Bản trước trống hoàn toàn ở K14–K20, nghĩa là MỌI video K14–K20 trong index đều
# là đáp án → retrieval trông tốt giả tạo ở các batch đó. Đã bổ sung.
_DIVERSITY = {
    "K01_V001",
    "K01_V002",
    "K01_V003",
    "K01_V004",
    "K01_V005",
    "K02_V001",
    "K02_V002",
    "K03_V001",
    "K03_V003",
    "K04_V001",
    "K04_V002",
    "K05_V001",
    "K05_V002",
    "K06_V001",
    "K06_V002",
    "K07_V001",
    "K07_V002",
    "K08_V001",
    "K09_V001",
    "K09_V002",
    "K10_V001",
    "K11_V001",
    "K12_V002",
    "K13_V001",
    "K14_V001",
    "K15_V001",
    "K16_V001",
    "K17_V001",
    "K18_V001",
    "K19_V001",
    "K20_V001",
    "L21_V001",
    "L21_V002",
    "L21_V003",
    "L22_V001",
    "L22_V002",
    "L23_V001",
    "L23_V002",
    "L24_V001",
    "L24_V002",
    "L25_V001",
    "L26_V001",
    "L26_V002",
    "L26_V003",
    "L27_V001",
    "L27_V002",
    "L28_V001",
    "L29_V001",
    "L30_V002",
    "L30_V003",
}

# ── Câu chưa có đáp án – bộ test KHÔNG đáp án ────────────────────────────────
# Dùng để soi kết quả bằng mắt / đối chiếu với đội khác, không tính được điểm.
_NO_ANSWER_QUERIES = ['R2-16', 'R2-19', 'R3-3', 'R3-5', 'R3-6', 'R3-9', 'R3-14', 'R3-19', 'R3-21', 'R3-22', 'R3-25', 'R3-27', 'R3-29', 'R3-30', 'R3-31', 'R3-34']

TEST_VIDEOS: set[str] = _GT_FRAME | _GT_VIDEO_ONLY | _NEAR_MISS | _DIVERSITY

print(f"[Chế độ] MODE = {MODE}")
print(f"[Video] Có đáp án đầy đủ  : {len(_GT_FRAME)} video")
print(f"[Video] Chỉ biết mã video : {len(_GT_VIDEO_ONLY)} video")
print(f"[Video] Tìm nhầm (nhiễu khó) : {len(_NEAR_MISS)} video")
print(f"[Video] Nhiễu thường      : {len(_DIVERSITY)} video")
print(f"[Video] TỔNG CỘNG         : {len(TEST_VIDEOS)} video"
      f"   ({len(_NO_ANSWER_QUERIES)} câu chưa có đáp án – phải MODE=FULL mới đo được)")

# ─── TRANSNETV2 ──────────────────────────────────────────────────────────────
TNV2_W        = 48
TNV2_H        = 27
CUT_THR       = 0.5
MIN_CUT_DIST  = 12
MIN_SCENE_LEN = 8
TNV2_CHUNK    = 200
TNV2_OVERLAP  = 25
TNV2_GDRIVE_ID = "1K7tpO6NMGVqCF30mwuyR-i2HZMA7Fsrg"

# ─── CHỌN KEYFRAME — LẤY MẪU THEO TỐC ĐỘ ─────────────────────────────────────
# Cách cũ: 4 frame mỗi cảnh, bất kể cảnh dài 8 frame hay 4500 frame.
#   → cảnh phỏng vấn 2,5 phút chỉ được 4 ảnh = 1 ảnh mỗi 37 giây. Bỏ sót gần hết.
# Cách mới: số ảnh TỈ LỆ với độ dài cảnh, có sàn và trần.
#
# Vì sao 2 giây: đồ hoạ tin tức và dòng chữ dưới màn hình tồn tại 4-6 giây, phụ đề
# 2-4 giây, cận cảnh vật thể 2-3 giây — nhịp 2 giây bắt được cả ba. Dày hơn thì
# ở MODE="FULL" sẽ ra ~1,8 triệu keyframe, phần trích đặc trưng không kham nổi.
# Khoảnh khắc dưới 1 giây thì không nhịp nào bắt được — đó là việc của TRAKE,
# phải quét dense riêng trong video đã chọn.
#
# Vì sao trần 40 chứ không phải 20: trần và bộ lọc trùng làm cùng một việc, mà
# bộ lọc làm tốt hơn vì nó nhìn nội dung. Cảnh phỏng vấn tĩnh 2 phút sinh 40 ảnh
# gần giống hệt nhau -> lọc trùng dọn sạch. Cảnh đua xe đạp 2 phút thì nội dung
# đổi thật -> giữ lại. Đặt trần thấp là cắt oan nhóm thứ hai, mà đó lại đúng là
# nhóm query khó nhất (R1-24, R1-25). Trần chỉ nên là van chống chạy loạn.
#   n = 2                              nếu cảnh ngắn hơn GIAY_HET_SAN
#   n = 2 + ⌈(dư ra) / GIAY_MOI_FRAME⌉   nếu dài hơn
#
# Quy ra frame:   30 fps → 8-50 : 2 ảnh | 51-110 : 3 | 111-170 : 4 | mỗi bậc +60
#                 25 fps → 8-42 : 2 ảnh | 43-92  : 3 |  93-142 : 4 | mỗi bậc +50
# Mốc theo GIÂY giống nhau ở mọi fps vì cả hai tham số đều tính bằng giây.
GIAY_HET_SAN   = 50 / 30  # ≈1,67 giây — hết vùng 2 ảnh, bắt đầu được 3 ảnh
GIAY_MOI_FRAME = 2.0      # sau mốc trên, cứ thêm 2 giây thì thêm 1 ảnh
N_MIN          = 2        # sàn — cảnh ngắn vẫn có đầu và cuối
N_MAX          = 40       # van an toàn; lọc trùng mới là thứ dọn cảnh tĩnh
MARGIN         = 3        # lùi vào mỗi đầu, tránh frame nằm ngay chỗ chuyển cảnh

# DEDUP_THR không còn dùng ở bước cắt. Theo paper §3.2.3, lọc trùng chạy SAU khi
# trích đặc trưng và dùng chính embedding nạp vào FAISS — tức thuộc SECTION 09/10.
# Giữ lại tham số ở đây để bước index dùng khi được bật lại.
DEDUP_THR     = 0.90
JPEG_QUALITY  = 90

# ─── ZIP ─────────────────────────────────────────────────────────────────────
ZIP_EVERY_GB  = 2.0
ZIP_DIR       = f"{OUTPUT_ROOT}/zips"
INDEX_DIR     = f"{OUTPUT_ROOT}/indexes"
PROGRESS_FILE = f"{OUTPUT_ROOT}/progress.json"

# ─── MODELS ──────────────────────────────────────────────────────────────────
SIGLIP_MODEL     = "google/siglip-large-patch16-384"
SIGLIP_BATCH     = 16
CLIP_MODEL       = "ViT-H-14"
CLIP_PRETRAIN    = "laion2b_s32b_b79k"
CLIP_BATCH       = 32
FAISS_SIGLIP_DIM = 1024
FAISS_CLIP_DIM   = 1024
NEIGHBOR_WINDOW  = 3

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in [OUTPUT_ROOT, ZIP_DIR, INDEX_DIR, LOCAL_KF_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"[Config] VIDEO_ROOT  : {VIDEO_ROOT}")
print(f"[Config] OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"[Config] LOCAL buffer: {LOCAL_KF_DIR}")
print(f"[Config] ZIP every   : {ZIP_EVERY_GB} GB")

# ── Load / Init progress ───────────────────────────────────────────────────────
def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        print(f"[Resume] {len(p.get('done_videos',[]))} videos done, "
              f"{p.get('zip_count',0)} zips created.")
        return p
    print("[Resume] No progress file found – starting fresh.")
    return {
        "done_videos": [],
        "zip_count": 0,
        "completed_zips": [],
        "buffer_videos": [],
        "buffer_size_bytes": 0,
        "total_frames_cut": 0,
        "index_done_siglip": [],
        "index_done_clip":   [],
        "index_phase_done":  False,
        # Feature extraction phase (Part 2)
        "done_ocr":     [],
        "done_asr":     [],
        "done_caption": [],
        "done_tags":    [],
        "done_objects": [],
        "features_merged": False,
        "last_updated": None,
    }

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()


[Chế độ] MODE = TEST
[Video] Có đáp án đầy đủ  : 63 video
[Video] Chỉ biết mã video : 4 video
[Video] Tìm nhầm (nhiễu khó) : 2 video
[Video] Nhiễu thường      : 50 video
[Video] TỔNG CỘNG         : 119 video   (16 câu chưa có đáp án – phải MODE=FULL mới đo được)
[Config] VIDEO_ROOT  : /content/drive/MyDrive/aic25/AIC_Video
[Config] OUTPUT_ROOT : /content/drive/MyDrive/aic26/output_demo/v004
[Config] LOCAL buffer: /content/kf_buffer
[Config] ZIP every   : 2.0 GB
[Resume] 0 videos done, 0 zips created.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02b – Buffer Recovery (quan trọng: phát hiện videos bị mất khi disconnect)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Khi Colab disconnect, /content/kf_buffer bị xóa.
# Videos đã mark done nhưng chưa kịp zip → keyframes mất.
# Section này phát hiện và đưa chúng về hàng đợi lại.
# %%

def _find_unzipped_videos() -> set[str]:
    """Tìm video names đã có trong các completed_zips."""
    found = set()
    for zname in PROGRESS.get("completed_zips", []):
        zpath = os.path.join(ZIP_DIR, zname)
        if not os.path.exists(zpath): continue
        with zipfile.ZipFile(zpath) as zf:
            for name in zf.namelist():
                parts = name.split("/")
                if parts: found.add(parts[0])
    return found

_zipped_videos = _find_unzipped_videos()
_buffer_lost   = [v for v in PROGRESS.get("buffer_videos", [])
                  if v not in _zipped_videos]

if _buffer_lost:
    print(f"[Recovery] Phát hiện {len(_buffer_lost)} videos trong buffer bị mất khi disconnect:")
    for v in _buffer_lost:
        print(f"  ↩ {v}")
    # Đưa về done_videos để cut lại
    for v in _buffer_lost:
        if v in PROGRESS["done_videos"]:
            PROGRESS["done_videos"].remove(v)
    PROGRESS["buffer_videos"]    = [v for v in PROGRESS["buffer_videos"]
                                    if v not in _buffer_lost]
    PROGRESS["buffer_size_bytes"] = 0
    save_progress(PROGRESS)
    print("[Recovery] Đã đưa về hàng đợi. Chúng sẽ được cắt lại trong Section 07.")
else:
    print("[Recovery] Buffer OK – không có video nào bị mất.")


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – List Videos (filter theo MODE)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

def list_all_videos(root: str) -> list[str]:
    vids = sorted(set(
        glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True) +
        glob.glob(os.path.join(root, "*.mp4"))
    ))
    if not vids:
        raise FileNotFoundError(f"Không tìm thấy video trong {root}")
    return vids

_all_raw      = list_all_videos(VIDEO_ROOT)
# Filter theo MODE
if MODE == "TEST":
    ALL_VIDEOS = [v for v in _all_raw if Path(v).stem in TEST_VIDEOS]
    _missing   = TEST_VIDEOS - {Path(v).stem for v in _all_raw}
    if _missing:
        print(f"[TEST] WARN: {len(_missing)} videos trong TEST_VIDEOS không tìm thấy trên Drive:")
        for m in sorted(_missing): print(f"  ✗ {m}")
else:  # FULL
    ALL_VIDEOS = _all_raw

done_set  = set(PROGRESS["done_videos"])
todo_list = [v for v in ALL_VIDEOS if Path(v).stem not in done_set]

total_gb = sum(os.path.getsize(v) for v in ALL_VIDEOS) / 1024**3
done_gb  = sum(os.path.getsize(v) for v in ALL_VIDEOS
               if Path(v).stem in done_set) / 1024**3

print(f"\n[Videos] MODE={MODE} | Total: {len(ALL_VIDEOS)} | Done: {len(done_set)} | Remaining: {len(todo_list)}")
print(f"[Size]   Total: {total_gb:.1f} GB | Done: {done_gb:.1f} GB")
print(f"[Sample] First 5 remaining:")
for v in todo_list[:5]:
    print(f"  {Path(v).stem}  ({os.path.getsize(v)/1024**2:.0f} MB)")

# ─── loại video không có trong bảng fps ─────────────────────────────────────
# fps là số CHỐT của BTC, không dò từ video. Video không có trong bảng thì không
# cắt được. Batch 1 chỉ có L21-L30; K01-K20 là batch 2, BTC chưa phát hành bảng.
# Lọc ở đây thay vì bỏ qua từng cái trong vòng lặp, để SO_VIDEO_THU lấy đúng
# video chạy được chứ không lấy nhầm video sẽ bị bỏ.
_thieu = [v for v in todo_list if Path(v).stem not in FPS_MAP]
todo_list = [v for v in todo_list if Path(v).stem in FPS_MAP]
if _thieu:
    _ten = [Path(v).stem for v in _thieu[:5]]
    _du  = " …" if len(_thieu) > 5 else ""
    print()
    print(f"[Bỏ]  {len(_thieu)} video không có trong bảng fps: {_ten}{_du}")
    print(f"[Còn] {len(todo_list)} video cắt được")

if SO_VIDEO_THU and len(todo_list) > SO_VIDEO_THU:
    todo_list = todo_list[:SO_VIDEO_THU]
    print(f"\n[THỬ] Giới hạn {len(todo_list)} video: "
          f"{[Path(v).stem for v in todo_list]}")
    print( "      Đặt SO_VIDEO_THU = 0 ở SECTION 02 rồi chạy lại ô này để cắt hết.")

if not todo_list:
    print("\n[✓] Tất cả video đã cut! Chuyển sang Section 09 để build index.")


[TEST] WARN: 1 videos trong TEST_VIDEOS không tìm thấy trên Drive:
  ✗ L24_V001

[Videos] MODE=TEST | Total: 118 | Done: 0 | Remaining: 118
[Size]   Total: 24.3 GB | Done: 0.0 GB
[Sample] First 5 remaining:
  K01_V001  (406 MB)
  K01_V002  (460 MB)
  K01_V003  (260 MB)
  K01_V004  (229 MB)
  K01_V005  (420 MB)


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 04 – Load TransNetV2 (auto-download weights nếu thiếu)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import torch
from transnetv2_pytorch import TransNetV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Device: {DEVICE}")

if not os.path.exists(CKPT_TNV2):
    print(f"[TransNetV2] Weights not found → downloading from GDrive...")
    import gdown
    gdown.download(id=TNV2_GDRIVE_ID, output=CKPT_TNV2, quiet=False)
    print(f"[TransNetV2] Downloaded → {CKPT_TNV2}")

def load_tnv2(ckpt_path: str):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m = TransNetV2()
    m.load_state_dict(state)
    return m.eval().to(DEVICE)

if todo_list:
    tnv2 = load_tnv2(CKPT_TNV2)
    print("[TransNetV2] Loaded OK")
else:
    print("[TransNetV2] Skipped (no videos remaining)")


[System] Device: cuda
[TransNetV2] Loaded OK


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# ⏸  TẠM TẮT — CLIP ViT-B/32 dùng cho lọc trùng
# ───────────────────────────────────────────────────────────────────────────────
# Bước lọc trùng đã dời sang SECTION 09/10 (dựng index), nơi nó dùng chính
# embedding SigLIP / CLIP ViT-H sẽ nạp vào FAISS — đúng như paper §3.2.3.
# Model này vì thế không còn cần ở bước cắt. Bỏ nó cũng tiết kiệm ~1 phút nạp
# và vài trăm MB VRAM mỗi phiên.
# ═══════════════════════════════════════════════════════════════════════════════

# # ═══════════════════════════════════════════════════════════════════════════════
# # SECTION 05 – Load CLIP ViT-B-32 for Alg.1 Dedup Filter
# # ═══════════════════════════════════════════════════════════════════════════════
# # %%
#
# import open_clip
#
# if todo_list:
#     clip_filter_model, _, clip_filter_prep = open_clip.create_model_and_transforms(
#         "ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE
#     )
#     clip_filter_model.eval()
#     print("[CLIP-filter] ViT-B-32 loaded for Alg.1 dedup")
# else:
#     print("[CLIP-filter] Skipped")
#

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 06 – Helper Functions (giữ nguyên từ nb_01)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import ffmpeg, cv2
from PIL import Image
from tqdm.auto import tqdm          # .auto → Colab hiện thanh đồ hoạ

# ──── Video Decode 48×27 cho TransNetV2 ────────────────────────────────────────
def decode_video_tnv2(path: str) -> np.ndarray:
    stream, _ = (
        ffmpeg.input(path)
        .output("pipe:", format="rawvideo", pix_fmt="rgb24", s=f"{TNV2_W}x{TNV2_H}")
        .run(capture_stdout=True, capture_stderr=True)
    )
    return np.frombuffer(stream, np.uint8).reshape([-1, TNV2_H, TNV2_W, 3])

def get_video_fps(path: str) -> float | None:
    """Trả fps theo bảng BTC. None nếu video không có trong bảng.

    KHÔNG dò fps từ file video nữa — bảng của BTC là số chốt, vì đáp án cuộc thi
    đánh số frame theo chính nó.
    """
    return FPS_MAP.get(Path(path).stem)

# ──── TransNetV2 Chunked (fix OOM cho video dài) ────────────────────────────────
def run_tnv2_chunked(frames: np.ndarray, model,
                     chunk: int = TNV2_CHUNK, ovl: int = TNV2_OVERLAP) -> np.ndarray:
    n = frames.shape[0]
    preds  = np.zeros(n, dtype=np.float32)
    counts = np.zeros(n, dtype=np.int32)
    start  = 0
    _buoc  = max(1, chunk - ovl)
    _tong  = max(1, -(-(n - ovl) // _buoc))        # số khúc, làm tròn lên
    _bar   = tqdm(total=_tong, desc="   dò chuyển cảnh", unit="khúc",
                  leave=False, dynamic_ncols=True)
    while start < n:
        end = min(start + chunk, n)
        t   = torch.from_numpy(frames[start:end].copy()).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            sp, _ = model(t)
            p = torch.sigmoid(sp).cpu().numpy().squeeze()
        p = np.atleast_1d(p).astype(np.float32)
        preds[start:end]  += p[:end-start]
        counts[start:end] += 1
        del t, sp, _
        torch.cuda.empty_cache()
        _bar.update(1)
        if end >= n: break
        start = end - ovl
    _bar.close()
    mask = counts > 0
    preds[mask] /= counts[mask]
    return preds

# ──── Scene Boundary ─────────────────────────────────────────────────────────────
def pick_cuts(prob: np.ndarray, thr=CUT_THR, min_d=MIN_CUT_DIST) -> list[int]:
    p, N = np.asarray(prob).ravel(), len(prob)
    mask, cuts = p >= thr, []
    i = 0
    while i < N:
        if mask[i]:
            j = i
            while j + 1 < N and mask[j+1]: j += 1
            cuts.append(i + int(np.argmax(p[i:j+1])))
            i = j + 1
        else:
            i += 1
    pruned = []
    for c in cuts:
        if not pruned or c - pruned[-1] >= min_d: pruned.append(c)
        elif p[c] > p[pruned[-1]]:               pruned[-1] = c
    return pruned

def cuts_to_scenes(cuts: list[int], n: int) -> list[tuple[int,int]]:
    cuts   = sorted(int(c) for c in cuts if 0 <= c < n)
    starts = [0]    + [c+1 for c in cuts]
    ends   = cuts   + [n-1]
    return [(s,e) for s,e in zip(starts,ends) if e-s+1 >= MIN_SCENE_LEN]

def sample_scene(s: int, e: int, fps: float) -> list[int]:
    """Lấy mẫu THEO TỐC ĐỘ: số ảnh tỉ lệ với độ dài cảnh.

    Kèm lùi biên MARGIN frame mỗi đầu. Frame cuối mỗi cảnh CHÍNH LÀ điểm cắt mà
    TransNetV2 chấm điểm cao nhất — với chuyển cảnh mềm đó là ảnh chồng đôi của
    hai cảnh. Lùi vào vài frame là tránh được.
    """
    L = e - s + 1
    # Bậc thang: N_MIN ảnh cho tới GIAY_HET_SAN, sau đó cứ GIAY_MOI_FRAME giây
    # thêm 1 ảnh. Dùng chia lấy trần trên số nguyên -(-a // b) để không dính
    # sai số dấu phẩy động ở đúng mốc chuyển bậc.
    _san  = max(1, int(round(fps * GIAY_HET_SAN)))      # 50 @30fps · 42 @25fps
    _buoc = max(1, int(round(fps * GIAY_MOI_FRAME)))    # 60 @30fps · 50 @25fps
    n = N_MIN if L <= _san else N_MIN + -(-(L - _san) // _buoc)
    n = max(N_MIN, min(N_MAX, n))
    if L <= n:
        return list(range(s, e + 1))
    m = min(MARGIN, (L - n) // 2)              # cảnh ngắn thì tự co biên lại
    return np.linspace(s + m, e - m, n, dtype=int).tolist()

# ──── Alg.1 Frame Filtering (Paper §3.2.3, cos > 0.9 → near-duplicate) ──────────
@torch.no_grad()
# ⚠ HÀM NÀY KHÔNG CÒN ĐƯỢC GỌI Ở BƯỚC CẮT.
# Giữ lại làm nguyên mẫu cho SECTION 09/10: ở đó thay `frames_48x27` bằng chính
# embedding SigLIP / CLIP ViT-H vừa tính cho từng keyframe, khỏi phải nhúng lại.
def alg1_dedup_filter(frames_48x27: np.ndarray,
                      selected_per_scene: list[list[int]],
                      model, preprocess,
                      S: float = DEDUP_THR) -> list[list[int]]:
    all_idx = list({i for scene in selected_per_scene for i in scene})
    if not all_idx: return [[] for _ in selected_per_scene]
    emb_map: dict[int, np.ndarray] = {}
    for start in tqdm(range(0, len(all_idx), 64), desc="   lọc trùng",
                      unit="lô", leave=False, dynamic_ncols=True):
        chunk_idx = all_idx[start:start+64]
        pils = [Image.fromarray(frames_48x27[i]) for i in chunk_idx]
        t    = torch.stack([preprocess(p) for p in pils]).to(DEVICE)
        f    = model.encode_image(t)
        f    = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()
        for ci, feat in zip(chunk_idx, f):
            emb_map[ci] = feat
    def cos(a, b): return float(np.dot(a, b))
    kept_all = []
    for scene in selected_per_scene:
        if not scene: kept_all.append([]); continue
        kept = [scene[0]]                      # frame đầu luôn giữ
        for t in range(1, len(scene)):
            e_t  = emb_map[scene[t]]
            drop = any(cos(e_t, emb_map[k]) >= S for k in kept)
            if not drop: kept.append(scene[t])

        # ── SÀN: mỗi cảnh giữ ít nhất N_MIN ảnh ──────────────────────────────
        # Bộ lọc so ảnh ở 48×27 nên thường coi frame đầu và frame cuối của một
        # cảnh ngắn là trùng nhau — dù dòng chữ dưới màn hình đã đổi hoàn toàn.
        # Bước lấy mẫu đã quyết định cảnh này chỉ đáng N_MIN ảnh, lọc tiếp là
        # lọc chồng lên chính nó. Bù lại bằng cách thêm dần frame KHÁC NHẤT so
        # với những cái đang giữ.
        _can = min(N_MIN, len(scene))
        while len(kept) < _can:
            _con  = [f for f in scene if f not in kept]
            _them = min(_con, key=lambda f: max(cos(emb_map[f], emb_map[k])
                                                for k in kept))
            kept.append(_them)

        kept_all.append(sorted(kept))          # trả về đúng thứ tự thời gian
    return kept_all

# ──── Save Keyframes (cv2 seek → ffmpeg fallback cho H.265/HEVC) ──────────────────
def save_keyframes(video_path: str, kept_per_scene: list[list[int]],
                   video_name: str, fps: float, out_dir: str,
                   quality: int = JPEG_QUALITY) -> list[dict]:
    import subprocess as _sp2
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    target_map: dict[int, tuple[int,int]] = {}
    for sid, scene in enumerate(kept_per_scene):
        for rank, fi in enumerate(scene):
            target_map[fi] = (sid, rank)
    if not target_map: return []
    sorted_targets = sorted(target_map.keys())

    # Method 1: cv2 seek (fast, H.264 CFR)
    local_tmp = f"/tmp/_aic_tmp_{video_name}.mp4"
    if "/content/drive" in video_path and not os.path.exists(local_tmp):
        shutil.copy2(video_path, local_tmp)
    read_path = local_tmp if os.path.exists(local_tmp) else video_path
    frame_buf: dict[int, np.ndarray] = {}
    cap = cv2.VideoCapture(read_path)
    if cap.isOpened():
        n_cv2 = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for fi in tqdm(sorted_targets, desc="   trích ảnh gốc", unit="frame",
                       leave=False, dynamic_ncols=True):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frm = cap.read()
            if ret: frame_buf[fi] = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
        cap.release()
        if not frame_buf:
            print(f"  [WARN] cv2 seek 0/{len(sorted_targets)} frames "
                  f"(cv2_total={n_cv2}, max_target={sorted_targets[-1]}) "
                  f"→ H.265/HEVC detected, switching to ffmpeg select fallback")

    # Method 2: ffmpeg select (reliable, works H.265/HEVC)
    if not frame_buf:
        ffmp_tmp = f"/tmp/_aic_ffout_{video_name}"
        Path(ffmp_tmp).mkdir(exist_ok=True)
        try:
            sel = "+".join(f"eq(n\\,{fi})" for fi in sorted_targets)
            cmd = ["ffmpeg", "-y", "-hwaccel", "auto", "-i", read_path,
                   "-vf", f"select='{sel}'", "-vsync", "vfr", "-q:v", "2",
                   os.path.join(ffmp_tmp, "f_%06d.jpg")]
            _sp2.run(cmd, capture_output=True, timeout=900)
            extracted = sorted(glob.glob(os.path.join(ffmp_tmp, "f_*.jpg")))
            for fi, fpath in zip(sorted_targets, extracted):
                if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
                    frame_buf[fi] = np.array(Image.open(fpath).convert("RGB"))
            print(f"  [FFmpeg] Extracted {len(frame_buf)}/{len(sorted_targets)} frames")
        except Exception as e:
            print(f"  [ERROR] ffmpeg fallback failed: {e}")
        finally:
            shutil.rmtree(ffmp_tmp, ignore_errors=True)

    if os.path.exists(local_tmp): os.remove(local_tmp)
    if not frame_buf:
        print(f"  [ERROR] Both methods failed for {video_name}"); return []

    first = next(iter(frame_buf.values()))
    H_img, W_img = first.shape[:2]
    meta_list = []
    for fi, (sid, rank) in tqdm(sorted(target_map.items(),
                                       key=lambda x: (x[1][0], x[1][1])),
                                desc="   ghi JPEG", unit="ảnh",
                                leave=False, dynamic_ncols=True):
        if fi not in frame_buf: continue
        fname    = f"{video_name}-{sid:04d}-{fi}.jpg"
        ts_ms    = int(fi / fps * 1000)
        ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                    f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
        fpath    = os.path.join(out_dir, fname)
        Image.fromarray(frame_buf[fi]).save(fpath, "JPEG", quality=quality)
        meta_list.append({
            "frame":         f"/static/images/{fname}",
            "url":           f"http://localhost:8000/static/images/{fname}",
            "name":          fname,
            "video":         video_name,
            "scene_id":      sid,
            "frame_idx":     fi,
            "fps":           fps,
            "width":         W_img,
            "height":        H_img,
            "timestamp_ms":  ts_ms,
            "timestamp_str": ts_str,
            "content": [], "ocr": [], "object": [], "color": [], "action": [],
            "faiss_id_clip":   -1,
            "faiss_id_siglip": -1,
            "neighbors_clip":  [],
        })
    return meta_list

# ──── ZIP & Upload to Drive ───────────────────────────────────────────────────────
def zip_and_upload(video_names: list[str], zip_id: int,
                   local_kf_dir: str, zip_dir: str) -> str:
    zip_name = f"AIC2026_frames_{zip_id:03d}.zip"
    zip_path = os.path.join(zip_dir, zip_name)
    print(f"  [Zip] Creating {zip_name} ({len(video_names)} videos)...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for vn in video_names:
            vdir = os.path.join(local_kf_dir, vn)
            if not os.path.isdir(vdir): continue
            for f in os.listdir(vdir):
                zf.write(os.path.join(vdir, f), arcname=f"{vn}/{f}")
    sz_mb = os.path.getsize(zip_path) / 1024**2
    print(f"  [Zip] {zip_name}: {sz_mb:.0f} MB in {time.time()-t0:.0f}s")
    for vn in video_names:
        vdir = os.path.join(local_kf_dir, vn)
        if os.path.isdir(vdir): shutil.rmtree(vdir)
    return zip_name

print("[SECTION 06] Helper functions OK")


[SECTION 06] Helper functions OK


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 07 – Main Cutting Loop  (auto-resume + auto-zip)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

if not todo_list:
    print("[SKIP] Tất cả video đã cut. Chuyển sang Section 09.")
else:
    BUFFER_META: list[dict] = []

    print(f"\n[Main Loop] {len(todo_list)} videos | Buffer trigger: {ZIP_EVERY_GB} GB\n")

    _bar_video = tqdm(todo_list, desc="Đang cắt", unit="video", dynamic_ncols=True)
    for video_path in _bar_video:
        video_name = Path(video_path).stem
        _bar_video.set_postfix_str(video_name)
        out_dir    = os.path.join(LOCAL_KF_DIR, video_name)
        t0         = time.time()
        print(f"\n  [{video_name}] START")

        fps = get_video_fps(video_path)
        if fps is None:
            print(f"  [BỎ QUA] {video_name}: không có trong fps_map.json"); continue

        try:
            frames_tnv2 = decode_video_tnv2(video_path)
        except Exception as e:
            print(f"  [ERROR] decode {video_name}: {e}"); continue

        n_frames = frames_tnv2.shape[0]
        print(f"  [1/4] Decoded 48×27: {n_frames} frames ({n_frames/fps:.0f}s)")

        prob   = run_tnv2_chunked(frames_tnv2, tnv2)
        cuts   = pick_cuts(prob)
        scenes = cuts_to_scenes(cuts, n_frames)
        print(f"  [2/4] TransNetV2: {len(cuts)} cuts → {len(scenes)} scenes")

        sampled = [sample_scene(s, e, fps) for s, e in scenes]
        print(f"  [3/4] Lấy mẫu: {sum(len(x) for x in sampled)} frame "
              f"từ {len(scenes)} cảnh — lọc trùng đã dời sang bước dựng index")
        del frames_tnv2; torch.cuda.empty_cache()

        meta = save_keyframes(video_path, sampled, video_name, fps, out_dir)
        _kt = f"{meta[0]['width']}×{meta[0]['height']}" if meta else "?"
        print(f"  [4/4] Saved {len(meta)} keyframes · {_kt} → {out_dir}")

        PROGRESS["done_videos"].append(video_name)
        PROGRESS["buffer_videos"].append(video_name)
        PROGRESS["total_frames_cut"] += len(meta)
        buf_size = sum(
            os.path.getsize(os.path.join(out_dir, f))
            for f in os.listdir(out_dir) if os.path.isfile(os.path.join(out_dir, f))
        ) if os.path.isdir(out_dir) else 0
        PROGRESS["buffer_size_bytes"] += buf_size
        BUFFER_META.extend(meta)
        save_progress(PROGRESS)
        print(f"  [{video_name}] DONE in {time.time()-t0:.0f}s | "
              f"Buffer: {PROGRESS['buffer_size_bytes']/1024**3:.2f} GB")

        if PROGRESS["buffer_size_bytes"] >= ZIP_EVERY_GB * 1024**3:
            zid      = PROGRESS["zip_count"] + 1
            zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
            PROGRESS["zip_count"]         = zid
            PROGRESS["completed_zips"].append(zip_name)
            PROGRESS["buffer_videos"]     = []
            PROGRESS["buffer_size_bytes"] = 0
            BUFFER_META = []
            save_progress(PROGRESS)
            print(f"  [Zip] ✓ {zip_name}")

    if PROGRESS["buffer_videos"]:
        zid      = PROGRESS["zip_count"] + 1
        zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
        PROGRESS["zip_count"]         = zid
        PROGRESS["completed_zips"].append(zip_name)
        PROGRESS["buffer_videos"]     = []
        PROGRESS["buffer_size_bytes"] = 0
        save_progress(PROGRESS)
        print(f"[Final Zip] ✓ {zip_name}")

    print(f"\n[✓] Cutting done! Total: {PROGRESS['total_frames_cut']:,} keyframes, "
          f"{PROGRESS['zip_count']} zips")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 07b · THỬ CẮT MỘT VIDEO — xem kết quả trước khi chạy hàng loạt
# ───────────────────────────────────────────────────────────────────────────────
# KHÔNG ghi progress.json, KHÔNG tạo zip, KHÔNG đụng gì trên Drive.
# Ảnh lưu tạm ở /content/thu_cat/ — mất khi Colab ngắt, đúng như mong muốn.
#
# Cần chạy trước: SECTION 00 → 06 (cài đặt, config, nạp TransNetV2 và CLIP lọc trùng).
# ═══════════════════════════════════════════════════════════════════════════════

VIDEO_THU     = "L21_V015"    # ← đổi tên video ở đây

# Xem ảnh: None = xem HẾT (chia nhiều trang), hoặc đặt số để xem mẫu trải đều.
SO_ANH_XEM    = None
ANH_MOI_TRANG = 48            # mỗi hình bấy nhiêu ảnh — quá 100 thì Colab ì
SO_COT        = 8

import matplotlib.pyplot as plt

# ─── đảm bảo model đã nạp ────────────────────────────────────────────────────
# SECTION 04 và 05 chỉ nạp model khi "todo_list" còn video chưa cắt. Nếu bạn đã
# chạy vòng lặp hàng loạt xong, todo_list rỗng → model không nạp → ô này lỗi
# NameError. Nạp bù ở đây cho chắc.
if "tnv2" not in dir():
    tnv2 = load_tnv2(CKPT_TNV2)
    print("[TransNetV2] nạp bổ sung")
# ─── tìm file video ──────────────────────────────────────────────────────────
_tim = (glob.glob(f"{VIDEO_ROOT}/**/{VIDEO_THU}.mp4", recursive=True) +
        glob.glob(f"{VIDEO_ROOT}/{VIDEO_THU}.mp4"))
assert _tim, f"Không thấy {VIDEO_THU}.mp4 trong {VIDEO_ROOT}"
_vpath = _tim[0]
_out   = f"/content/thu_cat/{VIDEO_THU}"
shutil.rmtree(_out, ignore_errors=True)

# ─── cắt ─────────────────────────────────────────────────────────────────────
_fps = get_video_fps(_vpath)
assert _fps is not None, f"{VIDEO_THU} không có trong fps_map.json"
print(f"[1/4] Đang giải mã {VIDEO_THU} xuống 48×27 — bước này ffmpeg chạy một mạch,")
print( "      không có thanh tiến độ. Video 20 phút mất khoảng 1-3 phút.")
_fr  = decode_video_tnv2(_vpath)
_n   = _fr.shape[0]
print(f"      {_n} frame · {_n/_fps/60:.1f} phút · {_fps:.1f} fps")

_prob   = run_tnv2_chunked(_fr, tnv2)
_cuts   = pick_cuts(_prob)
_scenes = cuts_to_scenes(_cuts, _n)
print(f"[2/4] TransNetV2: {len(_cuts)} điểm cắt → {len(_scenes)} cảnh "
      f"(bỏ {len(_cuts)+1-len(_scenes)} cảnh ngắn hơn {MIN_SCENE_LEN} frame)")

_moi = [sample_scene(s, e, _fps) for s, e in _scenes]                 # theo tốc độ
_cu  = [np.linspace(s, e, 4, dtype=int).tolist() for s, e in _scenes]  # cách cũ, để so

print(f"[3/4] Lấy mẫu: {sum(map(len,_moi))} frame — "
      f"không lọc trùng ở bước này (đã dời sang bước dựng index)")

del _fr; torch.cuda.empty_cache()
_meta = save_keyframes(_vpath, _moi, VIDEO_THU, _fps, _out)
print(f"[4/4] Đã lưu {len(_meta)} ảnh → {_out}\n")

# ─── thống kê ────────────────────────────────────────────────────────────────
_dai = np.array([e - s + 1 for s, e in _scenes])
print("=" * 68)
print(f"  ĐỘ DÀI CẢNH  ({len(_scenes)} cảnh)")
print("=" * 68)
for _ten, _v in (("ngắn nhất", _dai.min()), ("trung vị", int(np.median(_dai))),
                 ("trung bình", int(_dai.mean())), ("dài nhất", _dai.max())):
    print(f"  {_ten:<12}{_v:>7} frame   ({_v/_fps:>7.1f} giây)")

_spc_moi = np.array([len(x) for x in _moi])
_spc_cu  = np.array([len(x) for x in _cu])
print()
print("=" * 68)
print("  SO SÁNH HAI CÁCH LẤY MẪU")
print("=" * 68)
print(f"  {'':<26}{'cũ · 4/cảnh':>14}{'mới · theo tốc độ':>20}")
print(f"  {'-'*60}")
print(f"  {'Tổng ảnh':<26}{sum(map(len,_cu)):>14}{sum(map(len,_moi)):>20}")
print(f"  {'Ảnh ở cảnh NGẮN nhất':<26}{_spc_cu[_dai.argmin()]:>14}{_spc_moi[_dai.argmin()]:>20}")
print(f"  {'Ảnh ở cảnh DÀI nhất':<26}{_spc_cu[_dai.argmax()]:>14}{_spc_moi[_dai.argmax()]:>20}")
print(f"  {'Giãn cách cảnh dài nhất':<26}"
      f"{_dai.max()/_fps/_spc_cu[_dai.argmax()]:>13.1f}s"
      f"{_dai.max()/_fps/_spc_moi[_dai.argmax()]:>19.1f}s")
print()
print("  Phân bố số ảnh mỗi cảnh (cách mới):")
for _k in range(N_MIN, N_MAX + 1):
    _c = int((_spc_moi == _k).sum())
    if _c: print(f"    {_k:>3} ảnh  {'█'*min(_c, 46)} {_c}")

# ─── xem ảnh ─────────────────────────────────────────────────────────────────
def _thu_tu(f):
    """Sắp theo (số cảnh, số frame). Sort chuỗi sẽ để 0037-1000 đứng trước 0037-999."""
    _p = os.path.basename(f)[:-4].split("-")
    return (int(_p[-2]), int(_p[-1]))

_files = sorted(glob.glob(f"{_out}/*.jpg"), key=_thu_tu)

if SO_ANH_XEM and SO_ANH_XEM < len(_files):
    _files = [_files[i] for i in np.linspace(0, len(_files) - 1, SO_ANH_XEM, dtype=int)]
    print(f"\nXem {len(_files)} ảnh trải đều. Đặt SO_ANH_XEM = None để xem hết.\n")
else:
    _so_trang = -(-len(_files) // ANH_MOI_TRANG)
    print(f"\nXem toàn bộ {len(_files)} ảnh, chia thành {_so_trang} trang.\n")

for _t0 in range(0, len(_files), ANH_MOI_TRANG):
    _trang = _files[_t0:_t0 + ANH_MOI_TRANG]
    _r = -(-len(_trang) // SO_COT)
    _fig, _ax = plt.subplots(_r, SO_COT, figsize=(20, 2.0 * _r), squeeze=False)
    _o = _ax.ravel()
    for _a in _o:
        _a.axis("off")
    for _a, _f in zip(_o, _trang):
        _im = Image.open(_f); _im.thumbnail((240, 135))
        _a.imshow(np.array(_im))
        _p = os.path.basename(_f)[:-4].split("-")
        _a.set_title(f"c{int(_p[-2])} · f{_p[-1]} · {int(_p[-1])/_fps:.0f}s", fontsize=6.5)
    _fig.suptitle(f"{VIDEO_THU} — ảnh {_t0+1}–{_t0+len(_trang)} / {len(_files)}", fontsize=12)
    plt.tight_layout(); plt.show(); plt.close(_fig)

print("Nhãn dưới mỗi ảnh:  c<số cảnh> · f<số frame> · <giây trong video>")


[1/4] Đang giải mã L21_V015 xuống 48×27 — bước này ffmpeg chạy một mạch,
      không có thanh tiến độ. Video 20 phút mất khoảng 1-3 phút.
      39117 frame · 21.7 phút · 30.0 fps


[2/4] TransNetV2: 331 điểm cắt → 331 cảnh (bỏ 1 cảnh ngắn hơn 8 frame)
[3/4] Lấy mẫu: 1200 frame — không lọc trùng ở bước này (đã dời sang bước dựng index)


[4/4] Đã lưu 1200 ảnh → /content/thu_cat/L21_V015

  ĐỘ DÀI CẢNH  (331 cảnh)
  ngắn nhất        12 frame   (    0.4 giây)
  trung vị         83 frame   (    2.8 giây)
  trung bình      118 frame   (    3.9 giây)
  dài nhất        522 frame   (   17.4 giây)

  SO SÁNH HAI CÁCH LẤY MẪU
                               cũ · 4/cảnh   mới · theo tốc độ
  ------------------------------------------------------------
  Tổng ảnh                            1324                1200
  Ảnh ở cảnh NGẮN nhất                   4                   2
  Ảnh ở cảnh DÀI nhất                    4                  10
  Giãn cách cảnh dài nhất             4.3s                1.7s

  Phân bố số ảnh mỗi cảnh (cách mới):
      2 ảnh  ██████████████████████████████████████████████ 65
      3 ảnh  ██████████████████████████████████████████████ 154
      4 ảnh  ██████████████████████████████████████████████ 55
      5 ảnh  █████████████████ 17
      6 ảnh  ██████████ 10
      7 ảnh  █████████████ 13
      8 ảnh  ████

Nhãn dưới mỗi ảnh:  c<số cảnh> · f<số frame> · <giây trong video>


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 08 – Status Check (run bất kỳ lúc nào)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
zip_gb = sum(os.path.getsize(os.path.join(ZIP_DIR, z)) / 1024**3
             for z in p["completed_zips"]
             if os.path.exists(os.path.join(ZIP_DIR, z)))
print("=" * 60)
print(f"  PIPELINE STATUS  [{MODE} mode]")
print("=" * 60)
print(f"  Videos done        : {len(p['done_videos'])} / {len(ALL_VIDEOS)}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']} ({zip_gb:.1f} GB)")
print(f"  SigLIP indexed     : {len(p['index_done_siglip'])} / {p['zip_count']}")
print(f"  CLIP indexed       : {len(p['index_done_clip'])} / {p['zip_count']}")
print(f"  Index done         : {p['index_phase_done']}")
print(f"  OCR done           : {len(p.get('done_ocr', []))} videos")
print(f"  ASR done           : {len(p.get('done_asr', []))} videos")
print(f"  Caption done       : {len(p.get('done_caption', []))} videos")
print(f"  Features merged    : {p.get('features_merged', False)}")
print("=" * 60)


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 09 – Build SigLIP Index (stream through zips, resumable)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

from transformers import SiglipProcessor, SiglipModel
import faiss

p = load_progress()

# ─── chốt chặn: chưa có zip thì dừng ngay, đừng để SECTION 11 vỡ ────────────
if not p["completed_zips"]:
    print("=" * 72)
    print("  CHƯA CÓ FILE ZIP NÀO — SECTION 07 chưa chạy xong")
    print("=" * 72)
    print(f"  done_videos    : {len(p['done_videos'])}")
    print(f"  completed_zips : {p['completed_zips']}")
    _buf = os.listdir(LOCAL_KF_DIR) if os.path.exists(LOCAL_KF_DIR) else []
    print(f"  buffer local   : {len(_buf)} thư mục")
    print()
    if _buf:
        print("  → Đã cắt nhưng chưa nén. Chạy lại SECTION 07, nó sẽ nén nốt.")
    elif p["done_videos"]:
        print("  → Đã cắt nhưng ảnh mất do Colab ngắt. Chạy SECTION 02b để đưa")
        print("    các video đó về lại hàng đợi, rồi chạy lại SECTION 07.")
    else:
        print("  → Chạy SECTION 03 → 04 → 06 → 07 trước.")
    print("=" * 72)
    raise SystemExit

remaining_siglip = [z for z in p["completed_zips"] if z not in p["index_done_siglip"]]

if p["index_phase_done"]:
    print("[SKIP] Index phase already done.")
elif not remaining_siglip:
    print("[SKIP] SigLIP index already up to date.")
else:
    print(f"[SigLIP] Loading {SIGLIP_MODEL}...")
    siglip_proc  = SiglipProcessor.from_pretrained(SIGLIP_MODEL)
    siglip_model = SiglipModel.from_pretrained(SIGLIP_MODEL).to(DEVICE).eval()

    siglip_idx_path  = os.path.join(INDEX_DIR, "siglip.index")
    siglip_meta_path = os.path.join(INDEX_DIR, "siglip_mapping.json")
    if os.path.exists(siglip_idx_path):
        siglip_index = faiss.read_index(siglip_idx_path)
        with open(siglip_meta_path) as f: siglip_map = json.load(f)
        print(f"[SigLIP] Loaded existing: {siglip_index.ntotal} vectors")
    else:
        siglip_index = faiss.IndexFlatIP(FAISS_SIGLIP_DIM)
        siglip_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if os.path.exists(full_meta_path):
        with open(full_meta_path) as f: FULL_META = json.load(f)
        fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}
    else:
        FULL_META = []; fname_to_idx = {}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_siglip(img_paths: list[str], bs: int = SIGLIP_BATCH) -> np.ndarray:
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            imgs  = [Image.open(p).convert("RGB") for p in chunk]
            inp   = siglip_proc(images=imgs, return_tensors="pt").to(DEVICE)  # ← bỏ padding=True
            out   = siglip_model.get_image_features(**inp)
            feats = out.pooler_output if hasattr(out, 'pooler_output') else out  # ← fix
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().numpy())
        return np.vstack(all_feats).astype("float32")

    for zip_name in tqdm(remaining_siglip, desc="SigLIP indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [SigLIP] Embedding {len(img_paths)} images...")
        embeds   = embed_siglip(img_paths)
        start_id = siglip_index.ntotal
        siglip_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            siglip_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_siglip"] = faiss_id
            else:
                parts    = fname.rsplit("-", 2)
                video    = parts[0]
                scene_id = int(parts[1]) if len(parts) > 2 else 0
                fi       = int(parts[2].replace(".jpg","")) if len(parts) > 2 else 0
                # KHÔNG mặc định 25.0 — sai fps là sai toàn bộ timestamp.
                fps_v    = FPS_MAP[video]
                ts_ms    = int(fi / fps_v * 1000)
                ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                            f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
                entry = {
                    "faiss_id_siglip": faiss_id, "faiss_id_clip": -1,
                    "frame": f"/static/images/{fname}",
                    "url":   f"http://localhost:8000/static/images/{fname}",
                    "name":  fname, "video": video, "scene_id": scene_id,
                    "frame_idx": fi, "fps": fps_v, "timestamp_ms": ts_ms,
                    "timestamp_str": ts_str,
                    "content": [], "ocr": [], "object": [], "color": [], "action": [],
                    "neighbors_clip": [],
                }
                fname_to_idx[fname] = len(FULL_META)
                FULL_META.append(entry)
        faiss.write_index(siglip_index, siglip_idx_path)
        with open(siglip_meta_path, "w") as f: json.dump(siglip_map, f)
        with open(full_meta_path, "w") as f:   json.dump(FULL_META, f, indent=2)
        p["index_done_siglip"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {siglip_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[SigLIP] Done! Total: {siglip_index.ntotal}")


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 10 – Build CLIP ViT-H-14 Index
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()

# ─── chốt chặn: chưa có zip thì dừng ngay, đừng để SECTION 11 vỡ ────────────
if not p["completed_zips"]:
    print("=" * 72)
    print("  CHƯA CÓ FILE ZIP NÀO — SECTION 07 chưa chạy xong")
    print("=" * 72)
    print(f"  done_videos    : {len(p['done_videos'])}")
    print(f"  completed_zips : {p['completed_zips']}")
    _buf = os.listdir(LOCAL_KF_DIR) if os.path.exists(LOCAL_KF_DIR) else []
    print(f"  buffer local   : {len(_buf)} thư mục")
    print()
    if _buf:
        print("  → Đã cắt nhưng chưa nén. Chạy lại SECTION 07, nó sẽ nén nốt.")
    elif p["done_videos"]:
        print("  → Đã cắt nhưng ảnh mất do Colab ngắt. Chạy SECTION 02b để đưa")
        print("    các video đó về lại hàng đợi, rồi chạy lại SECTION 07.")
    else:
        print("  → Chạy SECTION 03 → 04 → 06 → 07 trước.")
    print("=" * 72)
    raise SystemExit

remaining_clip = [z for z in p["completed_zips"] if z not in p["index_done_clip"]]

if p["index_phase_done"]:
    print("[SKIP] Index phase already done.")
elif not remaining_clip:
    print("[SKIP] CLIP index already up to date.")
else:
    print(f"[CLIP] Loading {CLIP_MODEL} / {CLIP_PRETRAIN}...")
    clip_model_idx, _, clip_prep_idx = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN, device=DEVICE
    )
    clip_model_idx.eval()
    if 'clip_filter_model' in dir() and hasattr(clip_filter_model, 'parameters'):
        try: del clip_filter_model; torch.cuda.empty_cache()
        except: pass

    clip_idx_path  = os.path.join(INDEX_DIR, "clip.index")
    clip_meta_path = os.path.join(INDEX_DIR, "clip_mapping.json")
    if os.path.exists(clip_idx_path):
        clip_index = faiss.read_index(clip_idx_path)
        with open(clip_meta_path) as f: clip_map = json.load(f)
        print(f"[CLIP] Loaded existing: {clip_index.ntotal} vectors")
    else:
        clip_index = faiss.IndexFlatIP(FAISS_CLIP_DIM)
        clip_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if not ('FULL_META' in dir() and FULL_META):
        with open(full_meta_path) as f: FULL_META = json.load(f)
    fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_images_clip(img_paths: list[str], bs: int = CLIP_BATCH) -> np.ndarray:
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            pils  = [Image.open(p).convert("RGB") for p in chunk]
            t     = torch.stack([clip_prep_idx(img) for img in pils]).to(DEVICE)
            f     = clip_model_idx.encode_image(t)
            f     = f / f.norm(dim=-1, keepdim=True)
            all_feats.append(f.cpu().numpy())
        return np.vstack(all_feats).astype("float32")

    for zip_name in tqdm(remaining_clip, desc="CLIP indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [CLIP] Embedding {len(img_paths)} images...")
        embeds   = embed_images_clip(img_paths)
        start_id = clip_index.ntotal
        clip_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            clip_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_clip"] = faiss_id
        faiss.write_index(clip_index, clip_idx_path)
        with open(clip_meta_path, "w") as f: json.dump(clip_map, f)
        with open(full_meta_path, "w") as f:  json.dump(FULL_META, f, indent=2)
        p["index_done_clip"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {clip_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[CLIP] Done! Total: {clip_index.ntotal}")


In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 11 – Neighbor Lists + All Mapping Files
# ═══════════════════════════════════════════════════════════════════════════════
# %%

full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")

# ─── chốt chặn: file này do SECTION 09 tạo ra ───────────────────────────────
if not os.path.exists(full_meta_path):
    print("=" * 72)
    print("  CHƯA CÓ keyframe_metadata.json")
    print("=" * 72)
    print("  File này do SECTION 09 sinh ra khi nó index các file zip.")
    print("  Chuỗi phụ thuộc:  07 (cắt → zip)  →  09 (index → metadata)  →  11")
    print()
    print("  → Chạy SECTION 07, rồi 09, rồi quay lại ô này.")
    print("=" * 72)
    raise SystemExit

with open(full_meta_path) as f: FULL_META = json.load(f)
print(f"[Meta] Loaded {len(FULL_META)} keyframe entries")

video_groups: dict[str, list[int]] = defaultdict(list)
for i, m in enumerate(FULL_META):
    video_groups[m["video"]].append(i)
for video, indices in video_groups.items():
    video_groups[video] = sorted(indices, key=lambda i: FULL_META[i]["frame_idx"])

for video, indices in video_groups.items():
    n = len(indices)
    for pos, meta_idx in enumerate(indices):
        nbr_clip_ids = []
        for offset in range(-NEIGHBOR_WINDOW, NEIGHBOR_WINDOW + 1):
            if offset == 0: continue
            nbr_pos = pos + offset
            if 0 <= nbr_pos < n:
                nbr_meta = FULL_META[indices[nbr_pos]]
                cid = nbr_meta.get("faiss_id_clip", -1)
                if cid >= 0: nbr_clip_ids.append(cid)
        FULL_META[meta_idx]["neighbors_clip"] = nbr_clip_ids

with open(full_meta_path, "w") as f: json.dump(FULL_META, f, indent=2)
print(f"[Meta] Saved with neighbor lists")

# keyframes_list.json
kf_list = [f"{m['video']}-{m['frame_idx']}" for m in FULL_META]
with open(os.path.join(INDEX_DIR, "keyframes_list.json"), "w") as f:
    json.dump(sorted(set(kf_list)), f, indent=2)
print(f"[Meta] keyframes_list.json: {len(set(kf_list))} entries")

# image_mapping.json (backward compat: CLIP faiss_id → url)
image_mapping = {str(m.get("faiss_id_clip",-1)): m["url"]
                 for m in FULL_META if m.get("faiss_id_clip",-1) >= 0}
with open(os.path.join(INDEX_DIR, "image_mapping.json"), "w") as f:
    json.dump(image_mapping, f, indent=2)
print(f"[Meta] image_mapping.json: {len(image_mapping)} entries")

# fps_map.json
fps_unique = {m["video"]: m["fps"] for m in FULL_META}
with open(os.path.join(INDEX_DIR, "fps_map.json"), "w") as f:
    json.dump(fps_unique, f, indent=2)
print(f"[Meta] fps_map.json: {len(fps_unique)} videos")

p = load_progress(); p["index_phase_done"] = True; save_progress(p)


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 12 – Summary Part 1 (Baseline Complete)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
print("=" * 60)
print(f"  PART 1 COMPLETE  [{MODE} mode]")
print("=" * 60)
print(f"  Videos cut         : {len(p['done_videos'])}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']}")
print(f"  Index phase done   : {p['index_phase_done']}")
print()
for fname, desc in {
    "siglip.index":           "SigLIP FAISS (1152-dim)",
    "clip.index":             "CLIP FAISS (1024-dim)",
    "keyframe_metadata.json": "Rich metadata",
    "keyframes_list.json":    "UI submit form",
    "image_mapping.json":     "Backward compat",
    "fps_map.json":           "FPS per video",
}.items():
    fpath = os.path.join(INDEX_DIR, fname)
    if os.path.exists(fpath):
        print(f"  ✓ {fname:<30} {os.path.getsize(fpath)/1024**2:7.1f} MB  ({desc})")
    else:
        print(f"  ✗ {fname:<30}  MISSING")
print()
print("  → Tiếp theo: chạy nb_full_part2_features.py để extract OCR/ASR/Caption/Tags/Objects/Color")
